In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdChemReactions, Draw
import os
import sys
sys.path.append("/home/gridsan/yunsie/git_repo/chemprop")
import chemprop
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [2]:
def parity_plot(df_data, df_pred, target_types):
    for target_type in target_types:
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(1, 1, 1)

        true_list_all = df_data[target_type].to_list()
        pred_list_all = df_pred[target_type].to_list()
        uncertainty_list_all = df_pred[f'{target_type}_ensemble_uncal_var'].to_list()

        true_list = []
        pred_list = []
        uncertainty_list = []
        for i in range(len(true_list_all)):
            if not pd.isnull(true_list_all[i]):
                true_list.append(true_list_all[i])
                pred_list.append(pred_list_all[i])
                uncertainty_list.append(uncertainty_list_all[i])

        hb = ax.hexbin(
            true_list,
            pred_list,
            gridsize=100,
            cmap='plasma',
            mincnt=1
        )

        cb = plt.colorbar(hb, ax=ax)
        cb.set_label('Density', fontsize=15)

        axmin = min(min(true_list), min(pred_list)) - 0.1 * (max(true_list) - min(pred_list))
        axmax = max(max(true_list), max(pred_list)) + 0.1 * (max(true_list) - min(pred_list))
        ax.plot([axmin, axmax], [axmin, axmax], 'k--', lw=2, label='Ideal Prediction')

        ax.set_xlabel(f'Computed {target_type} (kcal/mol)', fontsize=15)
        ax.set_ylabel(f'Predicted {target_type} (kcal/mol)', fontsize=15)
        ax.set_title(f'Hexbin Plot for {target_type}', fontsize=18)

        R2 = r2_score(true_list, pred_list)
        mae = mean_absolute_error(true_list, pred_list)
        rmse = mean_squared_error(true_list, pred_list, squared=False)

        R2_uncertainty = np.std([r2_score(true_list, pred_list + np.random.normal(0, np.sqrt(u), len(pred_list))) for u in uncertainty_list])
        mae_uncertainty = np.std([mean_absolute_error(true_list, pred_list + np.random.normal(0, np.sqrt(u), len(pred_list))) for u in uncertainty_list])
        rmse_uncertainty = np.std([mean_squared_error(true_list, pred_list + np.random.normal(0, np.sqrt(u), len(pred_list)), squared=False) for u in uncertainty_list])

        ax.text(0.05, 0.85, f'$R^2$: {R2:.2f} ± {R2_uncertainty:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
        ax.text(0.05, 0.95, f'MAE: {mae:.2f} ± {mae_uncertainty:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
        ax.text(0.05, 0.90, f'RMSE: {rmse:.2f} ± {rmse_uncertainty:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')

        ax.tick_params(axis='both', which='major', labelsize=12)

        plt.tight_layout()
        plt.show()

In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def single_parity_plot(df_pred, df_data, data_name, pred_name):
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(1, 1, 1)

    true_list = df_data[data_name].to_list()
    pred_list = df_pred[pred_name].to_list()

    hb = ax.hexbin(
        true_list,
        pred_list,
        gridsize=100,
        cmap='plasma',
        mincnt=1
    )

    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Density', fontsize=15)

    axmin = min(min(true_list), min(pred_list)) - 0.1 * (max(true_list) - min(pred_list))
    axmax = max(max(true_list), max(pred_list)) + 0.1 * (max(true_list) - min(pred_list))
    ax.plot([axmin, axmax], [axmin, axmax], 'k--', lw=2, label='Ideal Prediction')

    ax.set_xlabel(f'Computed {data_name} (kcal/mol)', fontsize=15)
    ax.set_ylabel(f'Predicted {data_name} (kcal/mol)', fontsize=15)
    ax.set_title(f'Hexbin Plot for {data_name}', fontsize=18)

    R2 = r2_score(true_list, pred_list)
    mae = mean_absolute_error(true_list, pred_list)
    rmse = mean_squared_error(true_list, pred_list, squared=False)

    ax.text(0.05, 0.85, f'$R^2$: {R2:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.text(0.05, 0.95, f'MAE: {mae:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.text(0.05, 0.90, f'RMSE: {rmse:.2f}', transform=ax.transAxes, fontsize=12, verticalalignment='top')

    ax.tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout()
    plt.show()

In [4]:
def error_histogram(pred_data, true_data, target_types, target_pred):
    for target_type in target_types:
        predicted_values = pred_data[target_pred]
        computed_values = true_data[target_type]

        error = predicted_values - computed_values

        plt.figure(figsize=(5, 4))
        plt.hist(error, bins=100, density=True, color='blue', edgecolor='black')
        plt.title(f"Histogram of {target_pred} Error Probability", fontsize=14)
        plt.xlabel(f"{target_pred} Error (kcal/mol)", fontsize=12)
        plt.ylabel("Probability Density", fontsize=12)

        plt.tight_layout()
        plt.show()

In [5]:
def predicted_histogram(pred_data, target_types):
    for target_type in target_types:
        predicted_values = pred_data[target_type]

        error = predicted_values

        plt.figure(figsize=(5, 4))
        plt.hist(error, bins=40, density=True, color='blue', edgecolor='black')
        plt.title(f"Histogram of {target_type} Probability", fontsize=14)
        plt.xlabel(f"{target_type}", fontsize=12)
        plt.ylabel("Probability Density", fontsize=12)

        plt.tight_layout()
        plt.show()

In [6]:
def computed_histogram(true_data, target_types):
    for target_type in target_types:
        computed_values = true_data[target_type]

        error = computed_values

        plt.figure(figsize=(5, 4))
        plt.hist(error, bins=40, density=True, color='blue', edgecolor='black')
        plt.title(f"Histogram of {target_type} Probability", fontsize=14)
        plt.xlabel(f"{target_type} (kcal/mol)", fontsize=12)
        plt.ylabel("Probability Density", fontsize=12)

        plt.tight_layout()
        plt.show()

In [7]:
def molwt_histogram(pred_data, target_types):
    pred_data['Weights'] = pred_data[target_types].apply(lambda x: Chem.Descriptors.ExactMolWt(Chem.MolFromSmiles(x)))

    plt.figure(figsize=(5, 4))
    plt.hist(pred_data['Weights'], bins=15, density=True, color='blue', edgecolor='black')
    plt.title(f"Histogram of Molecular Weight", fontsize=14)
    plt.xlabel(f"Molecular Weight (g/mol)", fontsize=12)
    plt.ylabel("Density", fontsize=12)
    
    plt.tight_layout()
    plt.show()